# Data preparation

## Imports

In [1]:
1 + 1

2

In [2]:
import os
import sys

sys.path.append("..")

In [3]:
import numpy as np
import pandas as pd
import dill

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [4]:
# Comment next two lines to run in collab
%load_ext autoreload
%autoreload 2

from rl_trading.features.data_processing import (
    create_reverse_fx_tickers,
)

---
## Load raw historical data

In [5]:
# comment for collab
data_folder = "C:\\Users\\Ivan\\rl_trading\\data\\"
# data_folder = ""

In [6]:
historical_data = pd.read_parquet(f"{data_folder}FX_data.parquet.gzip")

In [7]:
for col in historical_data.columns:
    if col in ["ccy", "timestamp"]:
        continue
    historical_data[col] = historical_data[col].astype(float)

In [8]:
historical_data["ccy"] = historical_data["ccy"].str.upper()

In [9]:
# historical_data = historical_data.loc[historical_data["ccy"].isin(["EURUSD"]), :]

In [10]:
historical_data["ccy"].unique()

array(['EURJPY', 'EURUSD', 'SGDJPY', 'USDJPY', 'USDSGD'], dtype=object)

---
## Split into 3 parts

Leave first 3 years to train scaler

In [11]:
historical_data_scaler_train = historical_data.loc[
    historical_data["timestamp"] <= "2023-01-01 00:00:00", :
]

Take next 2 years for training

In [12]:
historical_data_train = historical_data.loc[
    (historical_data["timestamp"] >= "2023-01-02 00:00:00")
    & (historical_data["timestamp"] < "2023-12-01 00:00:00"),
    :,
]

Leave last month as validation set

In [13]:
historical_data_validate = historical_data.loc[
    (historical_data["timestamp"] >= "2023-12-01 00:00:00"), :
]

---
## Set close prices aside for state

In [14]:
def extract_close_prices(historical_data: pd.DataFrame) -> dict:
    """
    Extract close prices and transform to dict
    """
    historical_data = (
        pd.pivot_table(
            data=historical_data, index="timestamp", columns="ccy", values="close"
        )
        .ffill()
        .dropna()
    )
    
    historical_data = create_reverse_fx_tickers(historical_data)
    historical_data = historical_data.to_dict(orient="index")
    
    historical_data = {str(k): v for k, v in historical_data.items()}
    return historical_data

In [15]:
historical_prices_train = extract_close_prices(historical_data_train)
historical_prices_validate = extract_close_prices(historical_data_validate)

---
## Train and apply StandardScaler

In [16]:
normal_scaler = StandardScaler().fit(historical_data_scaler_train.drop(columns=["timestamp", "ccy"]))

In [17]:
def apply_scaler(normal_scaler: StandardScaler, hist_data: pd.DataFrame):
    features = normal_scaler.transform(hist_data.set_index(["timestamp", "ccy"]))
    multi_index = pd.MultiIndex.from_frame(hist_data[["timestamp", "ccy"]])
    features = pd.DataFrame(features, index=multi_index)
    return features.unstack("ccy").ffill().copy()

In [18]:
historical_data_scaler_train_scaled = apply_scaler(normal_scaler, historical_data_scaler_train)
historical_data_train_scaled = apply_scaler(normal_scaler, historical_data_train)
historical_data_validate_scaled = apply_scaler(normal_scaler, historical_data_validate)

---
## Train and apply PCA

In [19]:
pca_decomposition = PCA(n_components=26).fit(historical_data_scaler_train_scaled.dropna())

Leaving those that explain more than 0.5% of variance

In [20]:
pca_decomposition.explained_variance_ratio_

array([0.19819214, 0.15335513, 0.10747971, 0.10122628, 0.05185281,
       0.04992673, 0.04069634, 0.03171995, 0.02795071, 0.02013856,
       0.01868873, 0.0175397 , 0.01500795, 0.01419943, 0.01305394,
       0.01173868, 0.01025913, 0.00964107, 0.0085048 , 0.00791437,
       0.0075791 , 0.00658916, 0.00585703, 0.00571739, 0.00552378,
       0.00455795])

In [21]:
def apply_pca(pca_decomposition: PCA, hist_data: pd.DataFrame):
    features = pca_decomposition.transform(hist_data)
    final_dict = {}
    for i, date in enumerate(hist_data.index.to_list()):
        final_dict[str(date)] = features[i, :]
    return final_dict

In [22]:
historical_data_train_final = apply_pca(pca_decomposition, historical_data_train_scaled)
historical_data_validate_final = apply_pca(pca_decomposition, historical_data_validate_scaled)

---
## Save results

In [23]:
final_data = {
    "train_prices": historical_prices_train,
    "train_data": historical_data_train_final,
    "validate_prices": historical_prices_validate,
    "validate_data": historical_data_validate_final,
}

In [24]:
data_folder = "C:\\Users\\Ivan\\rl_trading\\data\\"

In [25]:
%%time
with open(f"{data_folder}preprocessed_data.pkl", "wb") as f:
    dill.dump(final_data, f, dill.HIGHEST_PROTOCOL)

CPU times: total: 58 s
Wall time: 1min
